<a href="https://colab.research.google.com/github/Subuktageen-Farooqi/ms_course_deeplearning/blob/main/ms_deeplearning_tutorial_09.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Object Detection: SSD & YOLO - Tensorflow Implementation

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
import numpy as np

# =========================
# SSD
# =========================

# Step 1: Define a simple backbone network for SSD
def build_backbone(input_shape=(300, 300, 3)):
    inputs = tf.keras.Input(shape=input_shape)
    x = layers.Conv2D(32, (3, 3), activation='relu', padding='same')(inputs)
    x = layers.MaxPooling2D((2, 2))(x)
    x = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(x)
    x = layers.MaxPooling2D((2, 2))(x)
    x = layers.Conv2D(128, (3, 3), activation='relu', padding='same')(x)
    base_model = models.Model(inputs, x, name="SSD_Backbone")
    return base_model

# Step 2: Define SSD prediction head for class and bounding box predictions
def ssd_head(x, num_classes, num_boxes):
    class_head = layers.Conv2D(num_boxes * num_classes, (3, 3), padding="same")(x)
    class_head = layers.Reshape((-1, num_classes))(class_head)

    box_head = layers.Conv2D(num_boxes * 4, (3, 3), padding="same")(x)  # 4 values for (x, y, w, h)
    box_head = layers.Reshape((-1, 4))(box_head)

    return class_head, box_head

# Step 3: Build the SSD model combining backbone, SSD heads, and multi-scale feature maps
def build_ssd_model(input_shape=(300, 300, 3), num_classes=21, num_boxes=6):
    # Backbone network
    base = build_backbone(input_shape)
    x = base.output  # Starting feature map from the backbone

    # Multi-scale feature maps for detection heads
    class_outputs, box_outputs = [], []
    for i in range(3):  # 3 scales for simplicity
        class_head, box_head = ssd_head(x, num_classes, num_boxes)
        class_outputs.append(class_head)
        box_outputs.append(box_head)

        # Downsample the feature map for next scale
        x = layers.Conv2D(128, (3, 3), strides=2, padding="same")(x)

    # Concatenate outputs across all scales
    classes_concat = layers.Concatenate(axis=1)(class_outputs)
    boxes_concat = layers.Concatenate(axis=1)(box_outputs)

    # Define the final SSD model
    ssd_model = models.Model(inputs=base.input, outputs=[classes_concat, boxes_concat])
    return ssd_model

# Instantiate SSD model
ssd_model = build_ssd_model()
ssd_model.summary()

# Step 4: Generate random dummy data and test the model
ssd_input = np.random.random((1, 300, 300, 3))
ssd_class_pred, ssd_box_pred = ssd_model.predict(ssd_input)

# Print shapes of predictions to confirm the output
print("\nSSD Class Prediction Shape:", ssd_class_pred.shape)
print("SSD Box Prediction Shape:", ssd_box_pred.shape)


# =========================
# YOLO
# =========================

# Step 1: Define a simple backbone for YOLO
def build_yolo_backbone(input_shape=(416, 416, 3)):
    inputs = tf.keras.Input(shape=input_shape)

    x = layers.Conv2D(32, (3, 3), strides=1, padding="same", activation="relu")(inputs)
    x = layers.MaxPooling2D((2, 2))(x)

    x = layers.Conv2D(64, (3, 3), strides=1, padding="same", activation="relu")(x)
    x = layers.MaxPooling2D((2, 2))(x)

    x = layers.Conv2D(128, (3, 3), strides=1, padding="same", activation="relu")(x)
    x = layers.MaxPooling2D((2, 2))(x)

    x = layers.Conv2D(256, (3, 3), strides=1, padding="same", activation="relu")(x)
    x = layers.MaxPooling2D((2, 2))(x)

    x = layers.Conv2D(512, (3, 3), strides=1, padding="same", activation="relu")(x)
    x = layers.MaxPooling2D((2, 2))(x)

    return models.Model(inputs, x, name="YOLO_Backbone")

# Step 2: Define YOLO head (inferred from tutorial text)
def yolo_head(x, num_classes=20, num_boxes=3):
    # output per box = 4 bbox coords + 1 confidence + num_classes
    output_channels = num_boxes * (5 + num_classes)
    output = layers.Conv2D(output_channels, (1, 1), padding="same")(x)
    return output

# Step 3: Combine backbone and YOLO head to build YOLO model
def build_yolo_model(input_shape=(416, 416, 3), num_classes=20, num_boxes=3):
    backbone = build_yolo_backbone(input_shape)
    x = backbone.output

    # YOLO head for detection
    output = yolo_head(x, num_classes, num_boxes)

    # Define the YOLO model
    yolo_model = models.Model(inputs=backbone.input, outputs=output)
    return yolo_model

# Instantiate YOLO model
yolo_model = build_yolo_model()
yolo_model.summary()

# Step 4: Generate dummy data for testing
# Generate a random dummy input image of shape (1, 416, 416, 3)
dummy_input = np.random.random((1, 416, 416, 3))

# Run a prediction on the dummy input
yolo_pred = yolo_model.predict(dummy_input)

# Print the shape of the output to verify
print("YOLO Prediction Shape:", yolo_pred.shape)


# Task 1: Object Detection YOLO SSD PyTorch Implementation

In [ ]:
import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# =========================
# SSD
# =========================


class SSDBackbone(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.features(x)


class SSDHead(nn.Module):
    def __init__(self, in_channels: int, num_classes: int, num_boxes: int):
        super().__init__()
        self.num_classes = num_classes
        self.num_boxes = num_boxes
        self.class_conv = nn.Conv2d(in_channels, num_boxes * num_classes, kernel_size=3, padding=1)
        self.box_conv = nn.Conv2d(in_channels, num_boxes * 4, kernel_size=3, padding=1)

    def forward(self, x):
        b = x.shape[0]
        class_logits = self.class_conv(x)
        box_reg = self.box_conv(x)

        class_logits = class_logits.permute(0, 2, 3, 1).contiguous()
        class_logits = class_logits.view(b, -1, self.num_classes)

        box_reg = box_reg.permute(0, 2, 3, 1).contiguous()
        box_reg = box_reg.view(b, -1, 4)
        return class_logits, box_reg


class SSDModel(nn.Module):
    def __init__(self, num_classes: int = 21, num_boxes: int = 6):
        super().__init__()
        self.num_classes = num_classes
        self.num_boxes = num_boxes
        self.backbone = SSDBackbone()

        # Reconstructed: matching the tutorial's 3-scale detection idea in PyTorch.
        self.heads = nn.ModuleList([SSDHead(128, num_classes, num_boxes) for _ in range(3)])
        self.downsample = nn.ModuleList(
            [nn.Conv2d(128, 128, kernel_size=3, stride=2, padding=1) for _ in range(2)]
        )

    def forward(self, x):
        x = self.backbone(x)
        class_outputs, box_outputs = [], []

        cls_0, box_0 = self.heads[0](x)
        class_outputs.append(cls_0)
        box_outputs.append(box_0)

        x = self.downsample[0](x)
        cls_1, box_1 = self.heads[1](x)
        class_outputs.append(cls_1)
        box_outputs.append(box_1)

        x = self.downsample[1](x)
        cls_2, box_2 = self.heads[2](x)
        class_outputs.append(cls_2)
        box_outputs.append(box_2)

        classes_concat = torch.cat(class_outputs, dim=1)
        boxes_concat = torch.cat(box_outputs, dim=1)
        return classes_concat, boxes_concat


# =========================
# YOLO
# =========================


class YOLOBackbone(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(128, 256, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(256, 512, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
        )

    def forward(self, x):
        return self.features(x)


class YOLOHead(nn.Module):
    def __init__(self, in_channels: int = 512, num_classes: int = 20, num_boxes: int = 3):
        super().__init__()
        self.output_channels = num_boxes * (5 + num_classes)
        # Reconstructed: single 1x1 conv prediction layer equivalent to the TF tutorial head.
        self.pred = nn.Conv2d(in_channels, self.output_channels, kernel_size=1)

    def forward(self, x):
        return self.pred(x)


class YOLOModel(nn.Module):
    def __init__(self, num_classes: int = 20, num_boxes: int = 3):
        super().__init__()
        self.backbone = YOLOBackbone()
        self.head = YOLOHead(in_channels=512, num_classes=num_classes, num_boxes=num_boxes)

    def forward(self, x):
        x = self.backbone(x)
        return self.head(x)


if __name__ == "__main__":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    ssd_transform = transforms.Compose([
        transforms.Resize((300, 300)),
        transforms.ToTensor(),
    ])

    yolo_transform = transforms.Compose([
        transforms.Resize((416, 416)),
        transforms.ToTensor(),
    ])

    train_dataset = datasets.VOCDetection(
        root="./data",
        year="2007",
        image_set="trainval",
        download=True,
        transform=transform
    )

    val_dataset = datasets.VOCDetection(
        root="./data",
        year="2007",
        image_set="test",
        download=True,
        transform=transform
    )

    train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, collate_fn=lambda x: x)
    val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False, collate_fn=lambda x: x)

    ssd_num_classes = 21
    ssd_num_boxes = 6
    yolo_num_classes = 20
    yolo_num_boxes = 3

    ssd_model = SSDModel(num_classes=21, num_boxes=6).to(device)
    yolo_model = YOLOModel(num_classes=20, num_boxes=3).to(device)

    with torch.no_grad():
        ssd_images, _ = next(iter(ssd_loader))
        ssd_images = ssd_images.to(device)
        ssd_class_pred, ssd_box_pred = ssd_model(ssd_images)
        print("SSD Input Shape:", tuple(ssd_images.shape))
        print("SSD Class Prediction Shape:", tuple(ssd_class_pred.shape))
        print("SSD Box Prediction Shape:", tuple(ssd_box_pred.shape))

        yolo_images, _ = next(iter(yolo_loader))
        yolo_images = yolo_images.to(device)
        yolo_pred = yolo_model(yolo_images)
        print("YOLO Input Shape:", tuple(yolo_images.shape))
        print("YOLO Prediction Shape:", tuple(yolo_pred.shape))

        assert ssd_class_pred.shape[-1] == ssd_num_classes
        assert ssd_box_pred.shape[-1] == 4
        assert yolo_pred.shape[1] == yolo_num_boxes * (5 + yolo_num_classes)

    print("All PyTorch shape checks passed.")

100%|██████████| 342M/342M [00:06<00:00, 50.3MB/s]


SSD Input Shape: (4, 3, 300, 300)
SSD Class Prediction Shape: (4, 44580, 21)
SSD Box Prediction Shape: (4, 44580, 4)
YOLO Input Shape: (4, 3, 416, 416)
YOLO Prediction Shape: (4, 75, 13, 13)
All PyTorch shape checks passed.


Task 02: Architecture Experiments (Alex Net, ResNet101, Mobile Net)

Task 03: Use your own dataset